# 🎬 Video Creator — AI Image Server

**Before running:** Runtime → Change runtime type → **T4 GPU**

Run cells **one at a time** — wait for each ✅ before moving to the next.

In [ ]:
# Cell 1 — Install (run once, ~1 min)
!pip install -q diffusers transformers accelerate flask flask-cors pyngrok
print('✅ Packages installed')

In [ ]:
# Cell 2 — Load model with CPU offload (prevents OOM crash on free T4)
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

# Clear any leftover GPU memory first
torch.cuda.empty_cache()

print('Loading model... (~5 min first time, instant after)')

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant='fp16',
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

# CPU offload — keeps VRAM under 4GB, prevents kernel crash
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing(1)

# Warmup pass to catch OOM early
print('Warming up...')
with torch.inference_mode():
    pipe('test', width=512, height=512, num_inference_steps=1, guidance_scale=1.0)

torch.cuda.empty_cache()
print(f'✅ Model ready — VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# Cell 3 — Flask API
import io, time, gc, torch
from flask import Flask, request, Response, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

# SDXL optimal sizes (multiples of 64)
SIZES = {
    'portrait':  (768, 1344),   # 9:16 Shorts
    'landscape': (1344, 768),   # 16:9 Video
    'square':    (1024, 1024),
}

@app.route('/health')
def health():
    vram = torch.cuda.memory_allocated() / 1e9
    return jsonify({'status': 'ok', 'model': 'sdxl', 'vram_gb': round(vram, 2)})

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data    = request.get_json(force=True)
        prompt  = data.get('prompt', 'cinematic scene, dramatic lighting')
        width   = int(data.get('width',  1080))
        height  = int(data.get('height', 1920))
        steps   = int(data.get('steps',  20))

        # Pick correct SDXL size
        if height > width:
            w, h = SIZES['portrait']
        elif width > height:
            w, h = SIZES['landscape']
        else:
            w, h = SIZES['square']

        full_prompt = f'{prompt}, cinematic, professional photography, sharp focus, high resolution, 8k'
        neg_prompt  = 'blurry, low quality, distorted, ugly, watermark, text, logo, nsfw'

        t0 = time.time()
        with torch.inference_mode():
            result = pipe(
                prompt=full_prompt,
                negative_prompt=neg_prompt,
                width=w, height=h,
                num_inference_steps=steps,
                guidance_scale=7.0,
            )

        elapsed = round(time.time() - t0, 1)
        print(f'[{elapsed}s] {prompt[:70]}')

        # Free memory after each generation
        gc.collect()
        torch.cuda.empty_cache()

        buf = io.BytesIO()
        result.images[0].save(buf, format='JPEG', quality=92)
        buf.seek(0)

        return Response(
            buf.read(),
            mimetype='image/jpeg',
            headers={'X-Generation-Time': str(elapsed)}
        )

    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        return jsonify({'error': 'OOM — try again'}), 503
    except Exception as e:
        print(f'Error: {e}')
        return jsonify({'error': str(e)}), 500

print('✅ Flask API ready')

In [ ]:
# Cell 4 — Start ngrok tunnel
# Paste your token from ngrok.com → Dashboard → Your Authtoken
NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'

from pyngrok import ngrok, conf
import threading

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()

# Start Flask in background
t = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)
)
t.daemon = True
t.start()

import time; time.sleep(1)  # let Flask start

tunnel = ngrok.connect(5000, 'http')
COLAB_URL = tunnel.public_url

# Print URL prominently — copy this into your .env.local
print('=' * 60)
print('🚀 SERVER LIVE')
print('=' * 60)
print()
print('Copy this into your .env.local:')
print()
print(f'  COLAB_IMAGE_URL={COLAB_URL}')
print()
print('=' * 60)
print('⚠️  Keep this tab open while generating videos')
print('   URL stays valid until you close Colab or ~8h')
print('=' * 60)

# Save URL to file so you can recover it if output scrolls away
with open('/content/colab_url.txt', 'w') as f:
    f.write(f'COLAB_IMAGE_URL={COLAB_URL}\n')
print()
print('URL also saved to /content/colab_url.txt')

In [ ]:
# Cell 5 — Check URL anytime (run this if you lose the URL)
try:
    with open('/content/colab_url.txt') as f:
        print(f.read())
except:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        print(f'COLAB_IMAGE_URL={t.public_url}')

In [ ]:
# Cell 6 — Test (optional, run to verify a real image is generated)
import requests
from IPython.display import Image as IPImage, display

print('Generating test image...')
r = requests.post(f'{COLAB_URL}/generate', json={
    'prompt': 'extreme close-up of a young Indian man counting rupee notes, dramatic shadow, cinematic',
    'width': 1080, 'height': 1920, 'steps': 20
}, timeout=120)

print(f'Status: {r.status_code} | Time: {r.headers.get("X-Generation-Time")}s')
if r.status_code == 200:
    display(IPImage(data=r.content, width=250))
    print('✅ Working!')
else:
    print(f'Error: {r.text}')